# SQ-GAN isolated speed benchmark

No benchmark updates count toward training. The active training run is not modified.
Each candidate restores the same successful 98-epoch checkpoint. Architecture, batch 1/GPU,
256x512 geometry, 800-image dataset and 150-epoch scheduler horizon remain fixed.
40 batches per candidate are timing probes, not full epochs or comparison results.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, time, hashlib, shutil, platform

RUN_MODE = "full"  # full training: 150 epochs x 3 stages x 2 channels
EPOCHS_PER_STAGE = 150
SESSION_HOURS = 9.5
SESSION_DEADLINE_UNIX = time.time() + SESSION_HOURS * 3600
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"  # legacy author torch.load behavior
STOP_AFTER_STAGE_EPOCHS = 150  # preserve the total stage target; session deadline handles chunking
CHANNELS = ["awgn", "rayleigh"]
SNR_DB = 10.0
PERFECT_CSI = True  # explicit Rayleigh adaptation assumption
IMAGE_HEIGHT = 256
IMAGE_WIDTH = 512
BATCH_SIZE = 1  # preserve author config

REPO = Path("/tmp/SQ-GAN")  # source, top-k checkpoints and images stay outside exported output
OUTPUT = Path("/tmp/sqgan_probe_source")
DIV2K_TRAIN = Path("/kaggle/input/notebooks/jagan028/div2k-dataset-generation-for-isr/Training/HR/hr_images")
DIV2K_VAL = Path("/kaggle/input/notebooks/jagan028/div2k-dataset-generation-for-isr/Validation/HR/hr_images")
KODAK_ROOT = Path("/kaggle/input/datasets/sherylmehta/kodak-dataset")
UPSTREAM_URL = "https://github.com/frapez1/SQ-GAN.git"
UPSTREAM_COMMIT = "0e5cdd7c0240b89749dad684645c53f783ac24cc"
BASE_CONFIG = "config/sqgan_cityscapes.yml"
OUTPUT.mkdir(parents=True, exist_ok=True)
print(json.dumps({"python":sys.version,"platform":platform.platform(),"run_mode":RUN_MODE,"upstream_commit":UPSTREAM_COMMIT}, indent=2))


In [ ]:
# Install the legacy Lightning generation required by the authors' two-optimizer training_step.
# Do not replace this with Lightning 2.x: it removed optimizer_idx and the old Trainer CLI APIs.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pytorch-lightning==1.9.5", "omegaconf==2.3.0",
                "torchmetrics==0.11.4", "einops", "easydict", "transformers==4.45.2"], check=True)
if not REPO.exists():
    subprocess.run(["git", "clone", UPSTREAM_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", UPSTREAM_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True)
import numpy as np
if not hasattr(np, "Inf"):
    np.Inf = np.inf  # runtime compatibility for NumPy 2; does not modify upstream source
for p in (REPO / "adaptations", REPO / "adaptations" / "sqgan_div2k_channel"):
    p.mkdir(parents=True, exist_ok=True)
(REPO / "adaptations" / "__init__.py").write_text("", encoding="utf-8")
(REPO / "adaptations" / "sqgan_div2k_channel" / "__init__.py").write_text("", encoding="utf-8")


In [ ]:
# Keep the adaptation code versioned in this notebook; the cloned author checkout stays untouched.
CHANNEL_SOURCE = 'from __future__ import annotations\n\nimport torch\nfrom torch import nn\n\n\nclass ComplexLatentChannel(nn.Module):\n    """Per-example unit-power complex AWGN or flat Rayleigh channel.\n\n    Input follows the author VQ module layout [batch, real_feature_dim, tokens].\n    A 256-D real code vector is serialized as 128 complex channel uses.\n    This is a DIV2K/AWGN/Rayleigh adaptation, not the original SQ-GAN pipeline.\n    """\n    def __init__(self, channel: str, snr_db: float = 10.0, perfect_csi: bool = True):\n        super().__init__()\n        self.channel = channel.lower()\n        if self.channel not in {"awgn", "rayleigh"}:\n            raise ValueError(f"channel must be awgn or rayleigh, got {channel!r}")\n        self.snr_db = float(snr_db)\n        self.perfect_csi = bool(perfect_csi)\n\n    def forward(self, z: torch.Tensor) -> torch.Tensor:\n        if z.ndim != 3:\n            raise ValueError(f"Expected [B,C,N] latent tensor, got {tuple(z.shape)}")\n        b, c, n = z.shape\n        if c % 2:\n            raise ValueError(f"Real latent dimension must be even, got {c}")\n        pairs = z.transpose(1, 2).contiguous().reshape(b, n, c // 2, 2)\n        power = pairs.square().sum(dim=-1).mean(dim=(1, 2), keepdim=True).clamp_min(1e-12)\n        scaled = pairs / power.sqrt().unsqueeze(-1)\n        x = torch.complex(scaled[..., 0], scaled[..., 1])\n        noise_power = 10.0 ** (-self.snr_db / 10.0)\n        sigma = (noise_power / 2.0) ** 0.5\n        noise = torch.complex(torch.randn_like(x.real), torch.randn_like(x.real)) * sigma\n        if self.channel == "rayleigh":\n            h = torch.complex(torch.randn_like(x.real), torch.randn_like(x.real)) / (2.0 ** 0.5)\n            y = h * x + noise\n            if self.perfect_csi:\n                x_hat = y * h.conj() / h.abs().square().clamp_min(1e-8)\n            else:\n                x_hat = y\n        else:\n            x_hat = x + noise\n        received = torch.stack((x_hat.real, x_hat.imag), dim=-1) * power.sqrt().unsqueeze(-1)\n        return received.reshape(b, n, c).transpose(1, 2).contiguous()\n\n\nclass ChannelizedVectorQuantizer(nn.Module):\n    """Keep author VQ and insert a differentiable channel after its STE output."""\n    def __init__(self, quantizer_config: dict, channel: str, snr_db: float = 10.0,\n                 perfect_csi: bool = True):\n        super().__init__()\n        from utils.utils import instantiate_from_config\n        self.quantizer = instantiate_from_config(quantizer_config)\n        self.channel = ComplexLatentChannel(channel, snr_db, perfect_csi)\n\n    def forward(self, *args, **kwargs):\n        quantized, commitment_loss, info = self.quantizer(*args, **kwargs)\n        return self.channel(quantized), commitment_loss, info\n'
DATASET_SOURCE = 'from __future__ import annotations\n\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch.utils.data import Dataset\nfrom torchvision import transforms\n\n\nclass DIV2KWithPseudoLabels(Dataset):\n    """Paired RGB and 19-class Cityscapes-ID pseudo-labels at author\'s 2:1 geometry."""\n    def __init__(self, image_root: str, label_root: str, size=(256, 512), train: bool = True):\n        self.image_root = Path(image_root)\n        self.label_root = Path(label_root)\n        if isinstance(size, int):\n            self.height = self.width = int(size)\n        else:\n            self.height, self.width = map(int, size)\n        self.train = bool(train)\n        exts = {".png", ".jpg", ".jpeg"}\n        images = sorted(p for p in self.image_root.rglob("*") if p.suffix.lower() in exts)\n        if not images:\n            raise FileNotFoundError(f"No images under {self.image_root}")\n        self.pairs = []\n        missing = []\n        for image in images:\n            mask = self.label_root / f"{image.stem}.png"\n            if mask.is_file():\n                self.pairs.append((image, mask))\n            else:\n                missing.append(str(mask))\n        if missing:\n            raise FileNotFoundError(f"Missing {len(missing)} pseudo masks; first: {missing[0]}")\n        self.image_norm = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))\n\n    def __len__(self):\n        return len(self.pairs)\n\n    def _resize_cover(self, image, mask):\n        scale = max(self.width / image.width, self.height / image.height)\n        new_size = (max(self.width, round(image.width * scale)), max(self.height, round(image.height * scale)))\n        image = image.resize(new_size, Image.Resampling.BICUBIC)\n        mask = mask.resize(new_size, Image.Resampling.NEAREST)\n        return image, mask\n\n    def __getitem__(self, idx):\n        image_path, mask_path = self.pairs[idx]\n        image = Image.open(image_path).convert("RGB")\n        mask = Image.open(mask_path).convert("L")\n        image, mask = self._resize_cover(image, mask)\n        if self.train:\n            x = int(np.random.randint(0, image.width - self.width + 1))\n            y = int(np.random.randint(0, image.height - self.height + 1))\n        else:\n            x = (image.width - self.width) // 2\n            y = (image.height - self.height) // 2\n        box = (x, y, x + self.width, y + self.height)\n        image, mask = image.crop(box), mask.crop(box)\n        if self.train and np.random.rand() < 0.5:\n            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)\n            mask = mask.transpose(Image.Transpose.FLIP_LEFT_RIGHT)\n        image_tensor = self.image_norm(transforms.ToTensor()(image))\n        ids = torch.from_numpy(np.asarray(mask, dtype=np.uint8).copy()).long()\n        if torch.any((ids != 255) & (ids > 18)):\n            raise ValueError(f"Expected Cityscapes IDs [0,18] or 255: {mask_path}")\n        ids[ids == 255] = 19\n        semantic = torch.nn.functional.one_hot(ids, num_classes=20).permute(2, 0, 1).float()\n        label_ids = ids.clone()\n        label_ids[label_ids == 19] = 0\n        label = label_ids.unsqueeze(0).float()\n        return {"image": image_tensor, "label": label, "semantic": semantic, "filename": image_path.name}\n'
ADAPTER = REPO / "adaptations" / "sqgan_div2k_channel"
(ADAPTER / "channel.py").write_text(CHANNEL_SOURCE, encoding="utf-8")
(ADAPTER / "dataset.py").write_text(DATASET_SOURCE, encoding="utf-8")







# BEGIN SESSION ORCHESTRATION
COMPAT_DIR = Path("/tmp/sqgan_runtime")
COMPAT_DIR.mkdir(parents=True, exist_ok=True)
(COMPAT_DIR / 'session_control.py').write_text('"""Notebook orchestration for the pinned author Lightning 1.9 training script."""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport os\nimport time\nfrom pathlib import Path\n\nimport pytorch_lightning as pl\nfrom pytorch_lightning.callbacks import Callback\n\n\nclass SessionControl(Callback):\n    def __init__(self):\n        super().__init__()\n        self.deadline = float(os.environ.get("SQGAN_SESSION_DEADLINE_UNIX", "inf"))\n        self.epoch_limit = int(os.environ.get("SQGAN_STOP_AFTER_STAGE_EPOCHS", "150"))\n        self.epoch_started = None\n\n    def on_train_start(self, trainer, pl_module):\n        expected = json.loads(os.environ.get("SQGAN_EXPECTED_RESUME", "null"))\n        rates = [[g["lr"] for g in opt.param_groups] for opt in trainer.optimizers]\n        actual = {"epoch_index": int(trainer.current_epoch), "global_step": int(trainer.global_step), "optimizer_learning_rates": rates}\n        if expected:\n            assert actual["global_step"] == expected["global_step"], (actual, expected)\n            assert len(trainer.optimizers) == expected["optimizer_states"], (actual, expected)\n            assert len(trainer.lr_scheduler_configs) == expected["lr_schedulers"], (actual, expected)\n            assert actual["epoch_index"] >= expected["saved_epoch_index"], (actual, expected)\n            for actual_groups, expected_groups in zip(rates, expected["optimizer_learning_rates"]):\n                assert len(actual_groups) == len(expected_groups)\n                assert all(math.isclose(a, b, rel_tol=1e-7, abs_tol=1e-12) for a, b in zip(actual_groups, expected_groups)), (actual, expected)\n            actual["resume_verified"] = True\n        else:\n            actual["resume_verified"] = False\n        if trainer.is_global_zero:\n            print("TRAINING STATE VERIFIED", json.dumps(actual), flush=True)\n            root = os.environ.get("SQGAN_STAGE_AUDIT_DIR")\n            if root:\n                destination = Path(root)\n                destination.mkdir(parents=True, exist_ok=True)\n                (destination / "resume_verified.json").write_text(json.dumps(actual, indent=2))\n\n    def on_train_epoch_start(self, trainer, pl_module):\n        self.epoch_started = time.time()\n\n    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, *args, **kwargs):\n        if trainer.is_global_zero and batch_idx % 100 == 0:\n            print("TRAIN PROGRESS", json.dumps({"epoch_index":int(trainer.current_epoch),"batch_index":batch_idx,"global_step":int(trainer.global_step),"training_batches":trainer.num_training_batches}), flush=True)\n\n    def on_train_epoch_end(self, trainer, pl_module, *args, **kwargs):\n        elapsed = time.time() - self.epoch_started if self.epoch_started else 0.0\n        completed = int(trainer.current_epoch) + 1\n        # Leave enough time for checkpoint serialization, mirroring and output upload.\n        local_stop = completed >= self.epoch_limit or time.time() + max(600, 1.5 * elapsed) >= self.deadline\n        stop = trainer.strategy.reduce_boolean_decision(local_stop, all=False)\n        trainer.should_stop = trainer.should_stop or stop\n        if trainer.is_global_zero:\n            record = {"completed_epochs": completed, "global_step": int(trainer.global_step), "epoch_seconds": elapsed, "session_seconds_remaining": max(0, self.deadline-time.time())}\n            for key, value in trainer.callback_metrics.items():\n                if getattr(value, "numel", lambda: 0)() == 1:\n                    record[key] = float(value.detach().cpu())\n            root = os.environ.get("SQGAN_STAGE_AUDIT_DIR")\n            if root:\n                with (Path(root) / "training_history.jsonl").open("a") as stream:\n                    stream.write(json.dumps(record) + "\\n")\n            print("EPOCH AUDIT", json.dumps(record), flush=True)\n            if stop:\n                print("NORMAL SESSION STOP REQUESTED at completed stage epoch", completed, flush=True)\n\n\ndef install():\n    if getattr(pl.Trainer, "_sqgan_session_control_installed", False):\n        return\n    original = pl.Trainer.fit\n\n    def fit(self, *args, **kwargs):\n        self.callbacks.append(SessionControl())\n        return original(self, *args, **kwargs)\n\n    pl.Trainer.fit = fit\n    pl.Trainer._sqgan_session_control_installed = True\n', encoding="utf-8")
(COMPAT_DIR / 'check_resume_contract.py').write_text('"""Verify Lightning checkpoint continuation on a tiny CPU model, not SQ-GAN."""\nimport json\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nif not hasattr(np, "Inf"):\n    np.Inf = np.inf\nos.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"\n\nimport torch\nimport pytorch_lightning as pl\nfrom torch.utils.data import DataLoader, TensorDataset\nfrom session_control import install\n\n\nclass Probe(pl.LightningModule):\n    def __init__(self):\n        super().__init__()\n        self.generator = torch.nn.Linear(2, 1)\n        self.discriminator = torch.nn.Linear(2, 1)\n\n    def training_step(self, batch, batch_idx, optimizer_idx):\n        layer = self.generator if optimizer_idx == 0 else self.discriminator\n        return layer(batch[0]).square().mean()\n\n    def configure_optimizers(self):\n        opts = [torch.optim.Adam(layer.parameters(), lr=0.01) for layer in [self.generator, self.discriminator]]\n        return opts, [{"scheduler": torch.optim.lr_scheduler.StepLR(opt, step_size=2, gamma=0.5), "interval": "step"} for opt in opts]\n\n\ndef main(root):\n    root = Path(root)\n    root.mkdir(parents=True, exist_ok=True)\n    install()\n    os.environ["SQGAN_SESSION_DEADLINE_UNIX"] = "inf"\n    os.environ["SQGAN_STOP_AFTER_STAGE_EPOCHS"] = "2"\n    os.environ["SQGAN_EXPECTED_RESUME"] = "null"\n    os.environ["SQGAN_STAGE_AUDIT_DIR"] = str(root / "first")\n    loader = DataLoader(TensorDataset(torch.ones(8, 2)), batch_size=2)\n    def trainer():\n        return pl.Trainer(accelerator="cpu", devices=1, max_epochs=4, logger=False, enable_checkpointing=False, enable_progress_bar=False, enable_model_summary=False)\n    first = trainer()\n    first.fit(Probe(), loader)\n    checkpoint = root / "probe.ckpt"\n    first.save_checkpoint(checkpoint)\n    raw = torch.load(checkpoint, map_location="cpu", weights_only=False)\n    assert raw["epoch"] == 2 or raw["epoch"] == 1\n    expected = {"saved_epoch_index": raw["epoch"], "global_step": raw["global_step"], "optimizer_states":len(raw["optimizer_states"]),"lr_schedulers":len(raw["lr_schedulers"]),"optimizer_learning_rates":[[g["lr"] for g in opt["param_groups"]] for opt in raw["optimizer_states"]]}\n    os.environ["SQGAN_EXPECTED_RESUME"] = json.dumps(expected)\n    os.environ["SQGAN_STOP_AFTER_STAGE_EPOCHS"] = "4"\n    os.environ["SQGAN_STAGE_AUDIT_DIR"] = str(root / "second")\n    second = trainer()\n    second.fit(Probe(), loader, ckpt_path=str(checkpoint))\n    assert second.current_epoch == 4, second.current_epoch\n    assert second.global_step > raw["global_step"]\n    os.environ["SQGAN_EXPECTED_RESUME"] = "null"\n    os.environ["SQGAN_SESSION_DEADLINE_UNIX"] = "0"\n    os.environ["SQGAN_STAGE_AUDIT_DIR"] = str(root / "deadline")\n    timed = trainer()\n    timed.fit(Probe(), loader)\n    assert timed.current_epoch == 1, timed.current_epoch\n    timed.save_checkpoint(root / "deadline_stop.ckpt")\n    timed_state = torch.load(root / "deadline_stop.ckpt", map_location="cpu", weights_only=False)\n    assert len(timed_state["optimizer_states"]) == len(timed_state["lr_schedulers"]) == 2\n    report = {"result":"PASS", "initial_global_step":raw["global_step"],"resumed_final_global_step":second.global_step,"resumed_final_epoch":second.current_epoch,"optimizer_and_scheduler_states_restored":True,"deadline_stop_and_checkpoint_verified":True}\n    (root / "resume_contract_pass.json").write_text(json.dumps(report, indent=2))\n    print("RESUME CONTRACT PASS", json.dumps(report), flush=True)\n\n\nif __name__ == "__main__":\n    import sys\n    main(sys.argv[1])\n', encoding="utf-8")

# Restore only the exported training state; the official source was freshly cloned above.
for input_root in Path("/kaggle/input").iterdir():
    for prior_out in input_root.rglob("sqgan_div2k_channel_results"):
        if prior_out.is_dir() and (prior_out / "stage_state").is_dir():
            shutil.copytree(prior_out, OUTPUT, dirs_exist_ok=True)
            print("RESTORED SAVED OUTPUT", prior_out, flush=True)
    for prior_logs in input_root.rglob("upstream_logs"):
        if prior_logs.is_dir():
            shutil.copytree(prior_logs, REPO / "logs", dirs_exist_ok=True)
            print("RESTORED UPSTREAM LOGS", prior_logs, flush=True)


In [ ]:
(COMPAT_DIR/"input_paths.py").write_text('"""Resolve Kaggle\'s namespaced and legacy mount layouts by actual file counts."""\nfrom pathlib import Path\n\ndef resolve_images(root,relative,count):\n    matches=[]\n    for folder in Path(root).glob(\'**/\'+relative):\n        if folder.is_dir():\n            files=[p for p in folder.rglob(\'*\') if p.suffix.lower() in {\'.png\',\'.jpg\',\'.jpeg\'}]\n            if len(files)==count: matches.append(folder)\n    assert len(matches)==1,f\'Expected one {relative} with {count} images; found {matches}\'\n    return matches[0]\n\ndef resolve_div2k(root=\'/kaggle/input\'):\n    return (resolve_images(root,\'Training/HR/hr_images\',800),\n            resolve_images(root,\'Validation/HR/hr_images\',100))\n')
sys.path.insert(0,str(COMPAT_DIR))
(COMPAT_DIR/"speed_probe.py").write_text('"""Isolated throughput probe. Never install in a production training session."""\nimport functools\nimport json\nimport os\nimport time\nfrom pathlib import Path\n\nimport torch\nimport pytorch_lightning as pl\nfrom pytorch_lightning.callbacks import Callback\n\n\nclass Probe(Callback):\n    def __init__(self):\n        self.times = []\n        self.waits = []\n        self.previous_end = None\n        self.start = None\n        self.resume = None\n        self.nonfinite = False\n\n    def on_train_start(self, trainer, module):\n        self.resume = {\'epoch_index\': int(trainer.current_epoch), \'global_step\': int(trainer.global_step),\n                       \'optimizers\': len(trainer.optimizers), \'schedulers\': len(trainer.lr_scheduler_configs),\n                       \'lr\': [[g[\'lr\'] for g in o.param_groups] for o in trainer.optimizers]}\n        expected = json.loads(os.environ[\'PROBE_EXPECTED\'])\n        assert self.resume[\'global_step\'] == expected[\'global_step\']\n        assert self.resume[\'optimizers\'] == self.resume[\'schedulers\'] == 2\n        assert self.resume[\'lr\'] == expected[\'lr\']\n        assert len(trainer.train_dataloader.dataset) == 800\n        assert trainer.num_training_batches == 400\n        torch.cuda.reset_peak_memory_stats()\n\n    def on_train_batch_start(self, trainer, module, batch, batch_idx):\n        torch.cuda.synchronize()\n        self.start = time.perf_counter()\n        if self.previous_end is not None:\n            self.waits.append(self.start - self.previous_end)\n\n    def on_before_backward(self, trainer, module, loss):\n        assert torch.isfinite(loss).all().item(), \'Non-finite probe loss\'\n\n    def on_train_batch_end(self, trainer, module, outputs, batch, batch_idx):\n        torch.cuda.synchronize()\n        end = time.perf_counter()\n        self.times.append(end-self.start)\n        self.previous_end = end\n        if len(self.times) >= 40:\n            trainer.should_stop = True\n\n    def on_train_end(self, trainer, module):\n        n = 10  # exclude cold-start warmup\n        compute = self.times[n:]\n        waits = self.waits[n-1:]\n        report = {\'variant\': os.environ[\'PROBE_VARIANT\'], \'rank\': trainer.global_rank,\n                  \'resume\': self.resume, \'samples_per_epoch\': 800, \'batches_per_full_epoch\': 400,\n                  \'batch_per_gpu\': 1, \'precision\': str(trainer.precision),\n                  \'batches_observed\': len(self.times), \'warmup_batches\': n,\n                  \'mean_compute_seconds\': sum(compute)/len(compute),\n                  \'mean_input_and_transfer_seconds\': sum(waits)/len(waits),\n                  \'mean_batch_seconds\': (sum(compute)+sum(waits))/len(compute),\n                  \'peak_allocated_bytes\': torch.cuda.max_memory_allocated(),\n                  \'all_model_parameters_finite\': all(torch.isfinite(p).all().item() for p in module.parameters()),\n                  \'benchmark_updates_count_as_training\': False}\n        assert report[\'all_model_parameters_finite\']\n        folder=Path(os.environ[\'PROBE_OUTPUT\']); folder.mkdir(parents=True,exist_ok=True)\n        (folder/f\'rank_{trainer.global_rank}.json\').write_text(json.dumps(report,indent=2))\n        print(\'SPEED PROBE RESULT\', json.dumps(report), flush=True)\n\n\ndef install():\n    from torch.utils.data import DataLoader\n    original_loader = DataLoader.__init__\n    @functools.wraps(original_loader)\n    def loader(self, *args, **kwargs):\n        kwargs[\'pin_memory\'] = os.environ.get(\'PROBE_PIN\') == \'1\'\n        if kwargs.get(\'num_workers\', 0) > 0:\n            kwargs[\'persistent_workers\'] = True\n            kwargs[\'prefetch_factor\'] = 2\n        return original_loader(self, *args, **kwargs)\n    DataLoader.__init__ = loader\n    cache_map = os.environ.get(\'PROBE_CACHE_MAP\')\n    if cache_map:\n        from adaptations.sqgan_div2k_channel.dataset import DIV2KWithPseudoLabels\n        original_init = DIV2KWithPseudoLabels.__init__\n        mapping=json.loads(Path(cache_map).read_text())\n        @functools.wraps(original_init)\n        def init(self,*args,**kwargs):\n            original_init(self,*args,**kwargs)\n            self.pairs=[tuple(Path(v) for v in mapping[str(pair[0])]) for pair in self.pairs]\n        DIV2KWithPseudoLabels.__init__=init\n    original_fit=pl.Trainer.fit\n    def fit(self,*args,**kwargs):\n        # Probe checkpoints and logger overhead are excluded consistently.\n        self.callbacks=[c for c in self.callbacks if type(c).__name__ not in {\'ModelCheckpoint\',\'CaptionImageLogger\'}]\n        self.callbacks.append(Probe())\n        return original_fit(self,*args,**kwargs)\n    pl.Trainer.fit=fit\n')
(COMPAT_DIR/"sitecustomize.py").write_text('import os, sys\nimport numpy as np\nif not hasattr(np, "Inf"): np.Inf=np.inf\nsys.path.insert(0,os.getcwd())\nimport pytorch_lightning.plugins as _plugins\nfrom pytorch_lightning.strategies import DDPStrategy\n_plugins.DDPPlugin=DDPStrategy\nfrom utils.logger import SetupCallback, CaptionImageLogger\nSetupCallback.on_fit_start=SetupCallback.on_pretrain_routine_start\ndelattr(SetupCallback,"on_pretrain_routine_start")\n_upstream_tensorboard=CaptionImageLogger._tensorboard\ndef _tensorboard(self,pl_module,images,batch_idx,split):\n    safe={k:v for k,v in images.items() if k not in ("ordered_vqidk","semantic_ordered_vqidk")}\n    return _upstream_tensorboard(self,pl_module,safe,batch_idx,split)\nCaptionImageLogger._tensorboard=_tensorboard\nfrom speed_probe import install\ninstall()\n')

import gc
import threading
import numpy as np
import torch
from PIL import Image
from omegaconf import OmegaConf
sys.path.insert(0,str(REPO))
os.chdir(REPO)
from adaptations.sqgan_div2k_channel.dataset import DIV2KWithPseudoLabels
from input_paths import resolve_div2k
DIV2K_TRAIN,DIV2K_VAL=resolve_div2k()
REPORT=Path('/kaggle/working/sqgan_speed_benchmark'); REPORT.mkdir(exist_ok=True)
checkpoint=next((REPO/'logs').glob('*sqgan_awgn_ssm*/checkpoints/last.ckpt'))
raw=torch.load(checkpoint,map_location='cpu',weights_only=False)
expected={'global_step':raw['global_step'],'lr':[[g['lr'] for g in opt['param_groups']] for opt in raw['optimizer_states']]}
assert len(raw['optimizer_states'])==len(raw['lr_schedulers'])==2
source_hash=hashlib.sha256(checkpoint.read_bytes()).hexdigest()
del raw; gc.collect()
source_cfg=OmegaConf.load(OUTPUT/'config_awgn_ssm.yaml')
cache=Path('/tmp/sqgan_resized_cache'); cache.mkdir(exist_ok=True)
mapping={}; equivalence=[]
for split in ('train','validation'):
    params=source_cfg.data.params[split].params
    params.image_root=str(DIV2K_TRAIN if split=='train' else DIV2K_VAL)
    params.label_root=str(OUTPUT/'pseudo_labels'/('div2k_train' if split=='train' else 'div2k_val'))
    dataset=DIV2KWithPseudoLabels(**OmegaConf.to_container(params,resolve=True))
    assert len(dataset)==(800 if split=='train' else 100)
    originals=list(dataset.pairs)
    for i,(image_path,mask_path) in enumerate(originals):
        with Image.open(image_path) as im, Image.open(mask_path) as ma:
            image,mask=dataset._resize_cover(im.convert('RGB'),ma.convert('L'))
        dest=cache/split; dest.mkdir(exist_ok=True)
        cp,mp=dest/f'{i}_rgb.png',dest/f'{i}_label.png'
        image.save(cp,compress_level=1); mask.save(mp,compress_level=1)
        mapping[str(image_path)]=[str(cp),str(mp)]
    # Exact equality after the same random crop/flip proves cache equivalence.
    for i in (0,len(dataset)//2,len(dataset)-1):
        np.random.seed(20261020+i); a=dataset[i]
        old=dataset.pairs[i]; dataset.pairs[i]=tuple(Path(v) for v in mapping[str(old[0])])
        np.random.seed(20261020+i); b=dataset[i]
        dataset.pairs[i]=old
        assert all(torch.equal(a[k],b[k]) for k in ('image','label','semantic'))
        equivalence.append({'split':split,'index':i,'all_tensors_exactly_equal':True})
    del dataset
(cache/'mapping.json').write_text(json.dumps(mapping))
(REPORT/'cache_equivalence.json').write_text(json.dumps(equivalence,indent=2))

variants=[('baseline_fp32',0,False,False,32),
          ('pinned_fp32',0,True,False,32),
          ('cached_fp32',0,True,True,32),
          ('workers2_cached_fp32',2,True,True,32),
          ('cached_fp16',0,True,True,16)]
results=[]
for name,workers,pin,cached,precision in variants:
    branch=REPO/'logs'/f'probe_{name}'
    (branch/'checkpoints').mkdir(parents=True,exist_ok=True)
    shutil.copy2(checkpoint,branch/'checkpoints/last.ckpt')
    cfg=copy.deepcopy(source_cfg)
    cfg.data.params.num_workers=workers
    cfgpath=Path('/tmp')/f'{name}.yaml'; OmegaConf.save(cfg,cfgpath)
    env=os.environ.copy()
    env.update(PYTHONPATH=str(COMPAT_DIR)+os.pathsep+str(REPO),
               PROBE_VARIANT=name,PROBE_OUTPUT=str(REPORT/name),
               PROBE_PIN='1' if pin else '0',PROBE_EXPECTED=json.dumps(expected))
    env.pop('PROBE_CACHE_MAP',None)
    if cached: env['PROBE_CACHE_MAP']=str(cache/'mapping.json')
    env['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD']='1'
    args=[sys.executable,'train.py','--base',str(cfgpath),'--mode','ssm',
          '--max_epochs','150','--gpus','0,1','--precision',str(precision),
          '--logtype','tensorboard','--num_sanity_val_steps','0','--enable_progress_bar','false',
          '--limit_val_batches','0','--resume',str(branch)]
    samples=[]; stop=threading.Event()
    def sample_gpus():
        while not stop.is_set():
            p=subprocess.run(['nvidia-smi','--query-gpu=index,utilization.gpu,memory.used',
                              '--format=csv,noheader,nounits'],capture_output=True,text=True)
            samples.append({'unix':time.time(),'csv':p.stdout.strip()})
            stop.wait(1)
    monitor=threading.Thread(target=sample_gpus,daemon=True); monitor.start()
    started=time.time()
    try:
        with (REPORT/f'{name}.log').open('w') as log:
            p=subprocess.run(args,cwd=REPO,env=env,stdout=log,stderr=subprocess.STDOUT,timeout=600)
        report_files=sorted((REPORT/name).glob('rank_*.json'))
        reports=[json.loads(f.read_text()) for f in report_files]
        result={'variant':name,'exit_code':p.returncode,'wall_seconds':time.time()-started,
                'rank_reports':reports,'passed':p.returncode==0 and len(reports)==2
                and all(r['batches_observed']==40 and r['all_model_parameters_finite'] for r in reports)}
    except subprocess.TimeoutExpired:
        # Killing DDP descendants is unsafe here; end this isolated Kaggle job.
        raise RuntimeError(f'{name} exceeded its probe budget; do not apply this candidate')
    finally:
        stop.set(); monitor.join(timeout=5)
    (REPORT/f'{name}_gpu_samples.json').write_text(json.dumps(samples))
    results.append(result)
    (REPORT/'results.json').write_text(json.dumps(results,indent=2))
    print('VARIANT RESULT',json.dumps(result),flush=True)
    shutil.rmtree(branch)
assert hashlib.sha256(checkpoint.read_bytes()).hexdigest()==source_hash
passing=[r for r in results if r['passed']]
baseline=next(r for r in passing if r['variant']=='baseline_fp32')
baseline_seconds=max(r['mean_batch_seconds'] for r in baseline['rank_reports'])
for r in passing:
    r['speedup_vs_baseline']=baseline_seconds/max(v['mean_batch_seconds'] for v in r['rank_reports'])
report={'source_checkpoint_sha256':source_hash,'source_unchanged':True,'results':results,
        'production_settings_changed':False,'benchmark_epochs_counted':0,
        'policy':'No automatic adoption. FP16 needs a full-epoch stability and quality check; workers need a full epoch coverage/resume check.'}
(REPORT/'benchmark_report.json').write_text(json.dumps(report,indent=2))
print('BENCHMARK COMPLETE',json.dumps(report),flush=True)
